In [3]:
import cv2
import numpy as np
import glob

# 1. SETTINGS
# Your board is 10x7 squares -> 9x6 internal corners
CHECKERBOARD = (9, 6) 
subpix_criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1)

objpoints = [] # 3D world space
imgpoints = [] # 2D image plane

# Prepare object points (0,0,0), (1,0,0), ... (8,5,0)
objp = np.zeros((1, CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[0,:,:2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

# 2. FILE LOADING
images = glob.glob("/home/wiebe/Desktop/Robotic/AE4317/testing_nn/data/calibration/*.jpg")

# Pre-processing tools
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))

print(f"Found {len(images)} images. Starting detection...")

for fname in images:
    img = cv2.imread(fname)
    if img is None: continue
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Pre-processing: Boost contrast to help the detector handle the yellow/low-light tint
    gray = clahe.apply(gray)
    
    # Robust detector: findChessboardCornersSB (Sector Based) 
    # It handles fisheye distortion much better than the standard version
    ret, corners = cv2.findChessboardCornersSB(gray, CHECKERBOARD, 
        cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE)

    if ret:
        objpoints.append(objp)
        imgpoints.append(corners)
        print(f"✔ Corners detected in: {fname.split('/')[-1]}")
    else:
        print(f"✘ Failed detection in: {fname.split('/')[-1]}")

# 3. CALIBRATION
N_OK = len(imgpoints)
if N_OK > 2: # You need at least 3 good images for a decent calibration
    K = np.zeros((3, 3))
    D = np.zeros((4, 1))
    
    # Using safe bitwise OR for flags
    # We remove 'FIX_SKEW' if your cv2 version is being picky and stick to standard defaults
    calib_flags = cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC | cv2.fisheye.CALIB_FIX_SKEW
    
    try:
        rms, _, _, _, _ = cv2.fisheye.calibrate(
            objpoints, 
            imgpoints, 
            gray.shape[::-1], 
            K, 
            D, 
            None, 
            None,
            calib_flags,
            (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-6)
        )
        print("\n--- Calibration Successful ---")
        print(f"RMS Error: {rms:.4f}")
        print(f"K Matrix (Intrinsic):\n{K}")
        print(f"D Matrix (Distortion):\n{D.flatten()}")
    except Exception as e:
        print(f"\nCalibration failed during math optimization: {e}")
else:
    print(f"\nNot enough images detected ({N_OK}/3). Try getting closer to the board or improving lighting.")

Found 94 images. Starting detection...
✔ Corners detected in: 36615153.jpg
✔ Corners detected in: 44548450.jpg
✔ Corners detected in: 79781484.jpg
✔ Corners detected in: 43948445.jpg
✔ Corners detected in: 41115136.jpg
✔ Corners detected in: 42048493.jpg
✔ Corners detected in: 34881880.jpg
✔ Corners detected in: 46115079.jpg
✔ Corners detected in: 39581814.jpg
✔ Corners detected in: 36848487.jpg
✔ Corners detected in: 35015183.jpg
✔ Corners detected in: 44181780.jpg
✔ Corners detected in: 36015191.jpg
✔ Corners detected in: 38815147.jpg
✔ Corners detected in: 41715115.jpg
✔ Corners detected in: 64048276.jpg
✔ Corners detected in: 36948508.jpg
✔ Corners detected in: 36715170.jpg
✔ Corners detected in: 38448477.jpg
✔ Corners detected in: 44315106.jpg
✔ Corners detected in: 35448532.jpg
✔ Corners detected in: 38948470.jpg
✔ Corners detected in: 59214987.jpg
✔ Corners detected in: 41948450.jpg
✔ Corners detected in: 35781846.jpg
✔ Corners detected in: 48148404.jpg
✔ Corners detected in: 42

In [ ]:
K Matrix (Intrinsic):
[[324.76411337   0.          22.64561962]
 [  0.         323.67794766 266.20294313]
 [  0.           0.           1.        ]]
D Matrix (Distortion):
[-0.0279868  -0.0362661   0.05859321 -0.03778534]